# BDB27 — Intra-drive regeneration: per-snap kinematic deficits → pass-rush performance

**This notebook is the whole interface.** Open it, read the pre-registration status in §3, set
`MODE` in §1 (and `DATA_DIR` only if the data is not in the usual place), then **Run All**. No
terminal, no CLI, no command to type.

**The design in one line.** The Combine-derived acceleration `a_0` is no longer the predictor; it is
each player's **baseline**. The modelled exposure is the **per-snap kinematic deficit** — the
current snap's peak acceleration as a fraction of that baseline (`deficit_ratio`) — which varies on
every snap instead of once per player.

**The two confirmatory questions, quoted from `TASK.md` (2026-11-04):**

> 1. **Primary (confirmatory):** Does the interaction between rolling kinematic load and inter-snap
>    rest predict a player's acceleration deficit on the current snap?
> 2. **Primary (confirmatory, second):** Do these transient, snap-level kinematic deficits predict
>    drops in pass-rush win rate and time-to-pressure better than absolute cumulative game load?

The exploratory question in `TASK.md` — at what rest-to-load ratio an elite starter's pressure
probability drops below a fresh rotational player's — is descriptive. Nothing in this notebook
tests it and no confirmatory claim is attached to it.

`notes/prereg.md` governs how those questions are analysed. Where it narrows a claim (it is a
living document until the full run), the prereg governs; §3 prints its declared status and its
current §12 text verbatim, and the headline never upgrades a status the prereg did not give.

**Provenance banner — read this before any number.**

* **`MODE = "full"`** — the final run. Every stage artefact is stamped `FULL`; only then may a
  number be quoted in the writeup. The headline block is live.
* **`MODE = "sample"`** — a plumbing run on the provided, truncated 2023 tracking sample
  (`2^20−1` rows). Every artefact is stamped `PLACEHOLDER_SAMPLE: UNVALIDATED SAMPLE OUTPUT`.
  **The headline prints `WITHHELD`**: statuses and reasons are shown, but no estimate is printed,
  because a sample number must never be mistaken for a finding.
* **A missing full-data file is not a crash.** `game_tracking_{2023,24,25}.csv` are not on this
  host. In full mode the stages that need them are reported as `PENDING FULL RUN`, the rest of the
  run continues, and the notebook names exactly which files are absent.

**What `05_validate` covers.** Exactly the Phase 5 items `TASK.md` names: the **within-player rest
permutation** (shuffle `inter_snap_rest_s` inside each rusher-game, refit the same Part-1 ladder,
compare the observed load × rest interaction with the permutation null) and the **context controls**
(garbage time; the final two minutes of each half). The third control `TASK.md` lists — obvious
passing situations — has **no definition anywhere in this repository**, so Conor skipped it by
decision on 2026-11-04 (D83, `notes/blockers.md` B-DL-IDR-01). It is emitted as
`SKIPPED_BY_DECISION` with its reason, including in the headline. A check that did not run is never
printed as a pass, and the retired spec's ceiling permutation is labelled `SUPERSEDED`, not run.

## 1. Configuration — edit only this cell

* `MODE="sample"` runs every stage on the data this host has, and stamps every artefact
  `UNVALIDATED SAMPLE OUTPUT`.
* `MODE="full"` is the final analysis. It requires the three full tracking files **and a frozen
  `notes/prereg.md`** (§3). While the prereg is `DRAFT` this notebook refuses to run full mode.
* `DATA_DIR=None` auto-detects the data directory: `/kaggle/input/...` if it exists, otherwise
  `<repo>/data/raw`. Set it only when the files live somewhere else.
* The **load window** is not a knob here: `rolling_kinematic_load_w_per_kg` is the integral of
  acceleration work over the prior **3** snaps (`config.yaml` → `bdb27_dl.idr_part1.windows.primary`;
  D80), with prior **1** and **5** snap windows declared as a sensitivity panel. The bootstrap cell
  reads those numbers from the config and prints them; they are never selected by looking at data.
* `FORCE=True` rebuilds every stage. `FORCE_STAGES` rebuilds only the named stages. `RUN_ONLY` runs
  only the named stages and reports the rest as `SKIPPED — checkpoint`.
* `CLEAN=True` is destructive but **inert unless `CLEAN_CONFIRM="DELETE"`**. It deletes only
  `outputs/dl/` and `data/interim/dl_checkpoints/`, never the parked LC/ceiling-v1 outputs.

In [ ]:
# ===================== CONFIG — edit only this cell =====================
MODE            = "sample"   # "sample" | "full"
DATA_DIR        = None       # None = auto-detect (/kaggle/input/... else <repo>/data/raw)
FORCE           = False      # True = force EVERY DL stage
FORCE_STAGES    = []         # e.g. ["03_game_snaps"]; [] = reuse checkpoints
RUN_ONLY        = []         # [] = every stage in DL_STAGE_ORDER; non-empty = only these stages
CLEAN           = False      # destructive; requires CLEAN_CONFIRM == "DELETE"
CLEAN_CONFIRM   = ""         # safety interlock
SHOW_FIGURES    = True       # render the optional diagnostic figures when the inputs exist
FIG_DPI         = 150

assert MODE in ("sample", "full"), MODE
assert DATA_DIR is None or isinstance(DATA_DIR, str), DATA_DIR
assert isinstance(FORCE, bool), FORCE
assert isinstance(FORCE_STAGES, (list, tuple)), FORCE_STAGES
assert all(isinstance(s, str) and s for s in FORCE_STAGES), FORCE_STAGES
assert isinstance(RUN_ONLY, (list, tuple)), RUN_ONLY
assert all(isinstance(s, str) and s for s in RUN_ONLY), RUN_ONLY
assert isinstance(CLEAN, bool), CLEAN
assert isinstance(CLEAN_CONFIRM, str), CLEAN_CONFIRM
assert isinstance(SHOW_FIGURES, bool), SHOW_FIGURES
assert isinstance(FIG_DPI, int) and FIG_DPI > 0, FIG_DPI

_EFFECTIVE_FORCE = ("ALL STAGES (FORCE=True)" if FORCE
                    else (list(FORCE_STAGES) or "none — reuse checkpoints"))
_CLEAN_ARMED = (CLEAN is True) and (CLEAN_CONFIRM == "DELETE")
print(f"mode              : {MODE}")
print(f"effective --force : {_EFFECTIVE_FORCE}")
print(f"RUN_ONLY          : {list(RUN_ONLY) or 'none — run every stage in DL_STAGE_ORDER'}")
print("CLEAN={} · CLEAN_CONFIRM={} — {}".format(
    CLEAN, "set" if CLEAN_CONFIRM else "empty",
    "ARMED: outputs/dl/ and the DL checkpoints will be deleted" if _CLEAN_ARMED
    else ("IGNORED: set CLEAN_CONFIRM='DELETE' to confirm" if CLEAN
          else "existing DL outputs are preserved")))
# ==========================================================================

## 2. Bootstrap — locate the repo, pick the data, write a scratch run-config, print the load window

This cell finds the repository by walking upward until it sees `config.yaml` and
`src/dl/dl_common.py`, picks the data directory (Kaggle-aware), writes
`bdb27_run/dl_config.yaml` with **that absolute data path**, and prints a short environment block:
Python version, repo commit, data directory, every required file present/absent, and the declared
load window read from the config.

A missing file is always named. Nothing is silently substituted.

In [ ]:
import copy
import importlib
import json
import shutil
import sys
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root() -> Path:
    """Walk upward from the notebook's working directory."""
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "config.yaml").exists() and (parent / "src" / "dl" / "dl_common.py").exists():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/dl/dl_common.py). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
SRC = REPO / "src"
LEGACY = SRC / "legacy_ceiling_v1"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(LEGACY) not in sys.path:
    sys.path.append(str(LEGACY))          # stage_runner lives with the preserved chain

import dl.dl_common as D                   # noqa: E402
import stage_runner                        # noqa: E402  (D46: reload stages from disk)


def _git_sha() -> str:
    """Short HEAD sha, or an explicit 'unavailable' — never a silent blank."""
    try:
        import subprocess
        return subprocess.check_output(
            ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
            text=True, stderr=subprocess.DEVNULL).strip()
    except Exception as exc:               # git absent, not a repo, or permission denied
        return f"unavailable ({type(exc).__name__})"


CFG_REPO = D.load_config(str(REPO / "config.yaml"))
DL_CFG = D.dl(CFG_REPO)

BASE_FILES = ["combine_tracking.csv", "combine_results.csv", "players.csv",
              "games.csv", "player_play.csv"]
_sample_key = DL_CFG["mode"]["sample"]["tracking_key"]
_full_key = DL_CFG["mode"]["full"]["tracking_key"]
_sample_value = DL_CFG["files"][_sample_key]
_full_value = DL_CFG["files"][_full_key]
SAMPLE_TRACKING = (_sample_value if isinstance(_sample_value, str)
                   else str(_sample_value[0]))
FULL_TRACKING = (list(_full_value) if isinstance(_full_value, (list, tuple))
                 else [_full_value])


def _score(d: Path) -> int:
    return sum((d / f).exists() for f in BASE_FILES)


def detect_data_dir(override):
    """Explicit override, else Kaggle input, else <repo>/data/raw. Never guesses a path silently."""
    if override:
        p = Path(override).expanduser().resolve()
        if _score(p):
            return p
        hits = sorted(p.rglob("combine_tracking.csv")) if p.is_dir() else []
        return hits[0].parent if hits else p
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        hits = sorted(kaggle.rglob("combine_tracking.csv"))
        if hits:
            candidates = sorted({h.parent for h in hits},
                                key=lambda d: (_score(d), len(str(d))), reverse=True)
            return candidates[0]
    return REPO / "data" / "raw"


DATA = detect_data_dir(DATA_DIR)

# ---- scratch run-config: only the data path (and the active mode default) change ----
RUN_DIR = REPO / "bdb27_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_CFG_PATH = RUN_DIR / "dl_config.yaml"
run_cfg = copy.deepcopy(CFG_REPO)
run_cfg.pop("_config_path", None)
run_cfg.setdefault("paths", {})["raw"] = str(DATA)
run_cfg["bdb27_dl"]["paths"]["raw"] = str(DATA)
run_cfg["bdb27_dl"]["mode"]["default"] = MODE
RUN_CFG_PATH.write_text(yaml.safe_dump(run_cfg, sort_keys=False), encoding="utf-8")
RUN_CFG = D.load_config(str(RUN_CFG_PATH))
DL_RUN = D.dl(RUN_CFG)


def _abs(path) -> Path:
    p = Path(path)
    return p if p.is_absolute() else (REPO / p)


_IDR_WINDOWS = DL_RUN["idr_part1"]["windows"]
LOAD_WINDOW = int(_IDR_WINDOWS["primary"])
LOAD_WINDOWS_SENSITIVITY = [int(w) for w in _IDR_WINDOWS["sensitivity"]]

OUTPUTS = _abs(DL_RUN["paths"]["outputs"])
FIGDIR = _abs(DL_RUN["paths"]["figures"])
CKPT = _abs(DL_RUN["paths"]["checkpoints"])
THRESH = _abs(DL_RUN["paths"]["thresholds"])
RUN_LOG = _abs(DL_RUN["paths"]["run_log"])
FIGDIR.mkdir(parents=True, exist_ok=True)

# ---- required-file inventory (list every absence) ---------------------------------
present = {f: (DATA / f).exists() for f in BASE_FILES}
SAMPLE_TRACKING_PRESENT = (DATA / SAMPLE_TRACKING).exists()
FULL_TRACKING_MISSING = [f for f in FULL_TRACKING if not (DATA / f).exists()]
FULL_TRACKING_READY = not FULL_TRACKING_MISSING
MISSING_BASE = [f for f in BASE_FILES if not present[f]]

# ---- CLEAN: only when explicitly armed; scoped to the DL chain --------------------
DL_CLEAN_DIRS = ("audit", "ceiling", "game", "models", "validation")
DL_CLEAN_FILES = ("thresholds.json",)
_CLEAN_ARMED = (CLEAN is True) and (CLEAN_CONFIRM == "DELETE")
if _CLEAN_ARMED:
    print("CLEAN=True confirmed — removing the DL chain outputs only:")
    for name in DL_CLEAN_DIRS:
        p = OUTPUTS / name
        n = sum(1 for f in p.rglob("*") if f.is_file()) if p.is_dir() else 0
        print(f"  {p.relative_to(REPO)}/  ({n} files)")
    for name in DL_CLEAN_FILES:
        p = OUTPUTS / name
        print(f"  {p.relative_to(REPO)}  ({1 if p.exists() else 0} files)")
    n_ckpt = len(list(CKPT.glob("*.json"))) if CKPT.is_dir() else 0
    print(f"  {CKPT.relative_to(REPO)}/  ({n_ckpt} DL checkpoints)")
    for name in DL_CLEAN_DIRS:
        shutil.rmtree(OUTPUTS / name, ignore_errors=True)
    for name in DL_CLEAN_FILES:
        try:
            (OUTPUTS / name).unlink()
        except FileNotFoundError:
            pass
    for f in OUTPUTS.glob("crash_*.txt"):
        f.unlink()
    if CKPT.is_dir():
        for f in CKPT.glob("*.json"):
            f.unlink()
elif CLEAN is True:
    print("CLEAN=True ignored: set CLEAN_CONFIRM='DELETE' to confirm. Outputs preserved.")


# ---- output readers used by the run and headline cells ----------------------------
def out_path(*parts) -> Path:
    return OUTPUTS.joinpath(*parts)


def read_csv(path, **kwargs) -> pd.DataFrame:
    path = Path(path)
    return pd.read_csv(path, **kwargs) if path.exists() else pd.DataFrame()


def read_json(path):
    """Parsed JSON, or None. A half-written file is 'absent', never a traceback."""
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None


def prov_of(df: pd.DataFrame, fallback: str = "") -> str:
    if len(df) and "provenance" in df.columns:
        return str(df["provenance"].iloc[0])
    return fallback


def prov_of_json(path) -> str:
    return str((read_json(path) or {}).get("provenance", ""))


def to_num(series):
    return pd.to_numeric(series, errors="coerce")


print(f"python          : {sys.version.split()[0]} ({sys.platform})")
print(f"repo            : {REPO}")
print(f"repo sha        : {_git_sha()}")
print(f"data dir        : {DATA}")
print(f"mode            : {MODE}   (force={FORCE}, clean={CLEAN})")
print("required files  :")
for f in BASE_FILES:
    print(f"  [{'ok ' if present[f] else 'MISSING'}] {f}")
print(f"  [{'ok ' if SAMPLE_TRACKING_PRESENT else 'MISSING'}] {SAMPLE_TRACKING}   (sample tracking)")
for f in FULL_TRACKING:
    print(f"  [{'ok ' if (DATA / f).exists() else 'MISSING'}] {f}   (full tracking)")
if MISSING_BASE:
    print("  !! missing required base file(s): " + ", ".join(MISSING_BASE))
    print("  !! the Combine/player stages cannot run until these are supplied.")
if not SAMPLE_TRACKING_PRESENT:
    print(f"  !! sample tracking absent: {SAMPLE_TRACKING} — sample-mode game-side work is limited.")
print(f"run config      : {RUN_CFG_PATH.relative_to(REPO)}")
print(f"outputs         : {OUTPUTS.relative_to(REPO)}/")
print(f"figures         : {FIGDIR.relative_to(REPO)}/")
print(f"load window     : rolling_kinematic_load_w_per_kg over the prior {LOAD_WINDOW} snaps "
      f"(declared sensitivity {LOAD_WINDOWS_SENSITIVITY}; never selected on data)")
print(f"checkpoints     : {CKPT.relative_to(REPO)}/")
if MODE == "full":
    if FULL_TRACKING_READY:
        print("full-data gate  : all full tracking files present")
    else:
        print("full-data gate  : PENDING FULL RUN — absent: " + ", ".join(FULL_TRACKING_MISSING))
        print("                  stages 01_audit, 03_game_snaps and 04_models will be skipped")

## 3. Pre-registration status — read from the file, and a hard gate

`notes/prereg.md` governs whether a confirmatory run may happen. This cell reads its **declared
status** at run time; the notebook does not assume it.

* **`Status: FROZEN`** → a full run is permitted, provided the data gate in §2 also passes.
* **`Status: DRAFT — NOT FROZEN`** — or any status that negates `FROZEN`, or no status line at all
  → **`MODE="full"` is refused here with an exception, before any stage runs.** There is no button
  to click through and no override flag: the file has to be frozen first.
* §12's **P1 rule** is printed **verbatim from the file**. Since D85 that text is **NO THRESHOLD**:
  P1 is an estimate with a 95% interval, not a tested hypothesis. This notebook never supplies,
  guesses or defaults a threshold the prereg does not declare.
* The stage gate (`dl_common.require_prereg`) applies the same rule: the first `**Status: ...**`
  block must affirm `FROZEN`, and a negated status such as `DRAFT — NOT FROZEN` is rejected even
  though it contains the substring `FROZEN`. The cell calls that same function, so the two cannot
  disagree.

In [ ]:
PREREG_PATH = REPO / "notes" / "prereg.md"
PREREG_EXISTS = PREREG_PATH.exists()
prereg_text = PREREG_PATH.read_text(encoding="utf-8") if PREREG_EXISTS else ""
# the declared status: the first "**Status: ...**" block, parsed exactly as the stage gate does
_status_line = D.prereg_declared_status(prereg_text) or ""


def _prereg_section(text: str, number: int):
    lines = text.splitlines()
    start = None
    for i, line in enumerate(lines):
        if line.strip().lower().startswith(f"## {number}."):
            start = i + 1
            break
    if start is None:
        return []
    out = []
    for line in lines[start:]:
        if line.strip().startswith("## "):
            break
        out.append(line)
    return out


_soi_lines = _prereg_section(prereg_text, 12)
_p1_idx = next((i for i, line in enumerate(_soi_lines) if "**P1" in line), None)
if _p1_idx is None:
    PREREG_P1_RULE_TEXT = "§12 P1 rule: NO THRESHOLD declared (D85); not restated verbatim here"
else:
    chunk = []
    for line in _soi_lines[_p1_idx:]:
        if chunk and line.lstrip().startswith("* "):
            break
        chunk.append(line.strip())
    PREREG_P1_RULE_TEXT = "§12 P1: " + " ".join(chunk)

# exactly the rule dl_common.require_prereg applies on the stage path
PREREG_FROZEN = PREREG_EXISTS and D.prereg_is_frozen(prereg_text)

print("PRE-REGISTRATION — notes/prereg.md")
print("-" * 88)
print(f"  file            : {PREREG_PATH.relative_to(REPO)} "
      f"({'present' if PREREG_EXISTS else 'MISSING'})")
print(f"  declared status : {_status_line or 'NO STATUS LINE FOUND'}")
print(f"  frozen?         : {PREREG_FROZEN}")
print(f"  {PREREG_P1_RULE_TEXT}")
print("  stage gate rule : dl_common.require_prereg requires the declared "
      "'**Status: ...**' block to affirm FROZEN")
print("  this cell uses  : D.prereg_is_frozen — the same function as the stage gate, "
      "so the two cannot disagree")
print("-" * 88)

if PREREG_FROZEN:
    print("PRE-REGISTRATION FROZEN — a full run is permitted (subject to the data gate in §2).")
else:
    print("FULL MODE IS BLOCKED.")
    print("  notes/prereg.md is DRAFT (not frozen). Until its open value is set and the file")
    print("  is frozen, `--mode full` is refused; sample mode remains available and stays")
    print("  stamped UNVALIDATED SAMPLE OUTPUT.")
    if MODE == "full":
        raise RuntimeError(
            "REFUSING TO RUN: MODE='full' while notes/prereg.md is DRAFT — NOT FROZEN. "
            "Freeze the prereg, then re-run. Set MODE='sample' to exercise the machinery "
            "meanwhile.")

## 4. The stage chain

`TASK.md` states the run order as `01_audit → 02_combine_ceiling → 00_thresholds → 03_game_snaps
→ 04_models → 05_validate`. The table below is in that order.

| # | stage | reads | writes |
|---|---|---|---|
| 1 | `01_audit` | `combine_*`, `players`, `games`, `player_play`, tracking probe | `outputs/dl/audit/` — `decision.json` (go/no-go), `SUMMARY.md`, items a–k (`i_icc.csv`, `c_snap_coverage.csv`, `d_null_semantics.csv`, `k_power.csv`, …) |
| 2 | `02_combine_ceiling` | `combine_tracking`, `combine_results`, `players` | `outputs/dl/ceiling/` — `player_ceiling.csv` (shrunken `a0`, SE), `summary.json`, `attempt_fits.csv` |
| 3 | `00_thresholds` | `config.yaml`, `notes/prereg.md` | `outputs/dl/thresholds.json` — grids, recency half-life, garbage-time margins, provenance/frozen flags |
| 4 | `03_game_snaps` | `player_play`, tracking, `thresholds.json`, `ceiling/player_ceiling.csv` | `outputs/dl/game/snaps/` — `snaps.csv` (one row per snap), `snaps_summary.json`, `SUMMARY.md` |
| 5 | `04_models` | `game/snaps/snaps.csv` | `outputs/dl/models/` — `part1_result.json`, `part2_result.json`, `model_result.json`; `outputs/dl/models/idr/` — `part1_idr.json`, `part2_idr.json`, `idr_terms.csv`, `idr_windows.csv`, `part2_idr_folds.csv` |
| 6 | `05_validate` | `04` + `03` outputs, `thresholds.json` | `outputs/dl/validation/` — `validation_result.json`, `rest_permutation_null.csv`, `context_controls.csv`, `context_sensitivity.csv`, `SUMMARY.md` |

**IDR names to look for.** Phase 3 writes `inter_snap_rest_s`, `rest_gap_plays`,
`rest_sequence_complete`, `rep_kinematic_load_w_per_kg`, `rolling_kinematic_load_w_per_kg`,
`load_window_n`, `a0_snap_yds2`, `tau_snap_s`, `fit_ok`, `baseline_a0_yds2`, `baseline_a0_se` and
`deficit_ratio`. Phase 4's IDR results live in the `models/idr/` sub-block: `part1_idr.json`
(`primary` + `panel`, the load × rest interaction with its rung ladder) and `part2_idr.json` (the
deficit-vs-absolute held-out comparison). Phase 5's result is `validation_result.json` →
`rest_permutation` (+ `context_controls`).

**Executable order.** This notebook keeps `00_thresholds` first in `DL_STAGE_ORDER`, because
`03/04/05` read `thresholds.json` and the stages are independent of `01`/`02` only up to that
point. The six stages and their contents are exactly `TASK.md`'s; only that one ordering detail
differs, and it is pinned by `tests/test_dl_notebook_contract.py`.

Each stage is invoked through its **own `main()`**, in-process, with `stage_runner.run_stage`,
which purges cached bytecode and reloads the stage and its helpers from disk before every call. A
`git pull` therefore cannot leave stale code in the kernel.

A stage that needs the absent full tracking files is reported as **`PENDING FULL RUN`** and is not
executed — it is not allowed to produce a `FULL`-stamped artefact from the sample. A genuine
exception is reported as `FAILED` with its traceback tail and crash-file path; the loop does not
abort.

In [ ]:
# ---- the DL chain; exactly the six stages TASK.md specifies ---------------------------
DL_STAGE_ORDER = ["00_thresholds", "01_audit", "02_combine_ceiling",
                  "03_game_snaps", "04_models", "05_validate"]
STAGE_MODULES = {
    "00_thresholds": "dl.stage_00_thresholds",
    "01_audit": "dl.stage_01_audit",
    "02_combine_ceiling": "dl.stage_02_ceiling",
    "03_game_snaps": "dl.stage_03_snaps",
    "04_models": "dl.stage_04_models",
    "05_validate": "dl.stage_05_validate",
}
#: Stages that cannot be honest without the full tracking files.
FULL_DATA_STAGES = {"01_audit", "03_game_snaps", "04_models", "05_validate"}

DL_HELPERS = (
    "dl.dl_common", "dl.dl_ceiling_core", "dl.dl_load", "dl.dl_snaps",
    "dl.dl_blockers", "dl.dl_models_core", "dl.dl_survival",
    "dl.dl_audit_ab", "dl.dl_audit_ce", "dl.dl_audit_f", "dl.dl_audit_power",
    "dl.dl_phase3", "dl.dl_idr_models", "dl.dl_idr_part2", "dl.dl_idr_validate",
)

# Crash reports land next to each stage's own outputs (outputs/dl/<sub>/crash_*.txt).
stage_runner.run_diagnostics.STAGE_OUTPUTS.update({
    "dl.stage_00_thresholds": ("", ("thresholds.json",)),
    "dl.stage_01_audit": ("audit", ("decision.json",)),
    "dl.stage_02_ceiling": ("ceiling", ("player_ceiling.csv", "summary.json")),
    "dl.stage_03_snaps": ("game/snaps", ("snaps.csv", "snaps_summary.json")),
    "dl.stage_04_models": ("models", ("model_result.json",)),
    "dl.stage_05_validate": ("validation", ("validation_result.json",)),
})


def _runlog_rows() -> int:
    if not RUN_LOG.exists():
        return 0
    try:
        return len(pd.read_csv(RUN_LOG))
    except Exception:
        return 0


def _peak_mb_after(offset: int, stage: str):
    """The stage's own peak RSS from outputs/run_log.csv (None when it was skipped)."""
    if not RUN_LOG.exists():
        return None
    try:
        frame = pd.read_csv(RUN_LOG)
    except Exception:
        return None
    new = frame.iloc[offset:]
    hit = new[(new["stage"] == stage) & (new["mode"] == MODE)]
    if not len(hit):
        return None
    values = pd.to_numeric(hit["peak_rss_mb"], errors="coerce").dropna()
    return float(values.iloc[-1]) if len(values) else None


def _run_dl_stage(name: str, extra_args):
    return stage_runner.run_stage(
        STAGE_MODULES[name], [*extra_args, "--config", str(RUN_CFG_PATH)],
        outputs_root=OUTPUTS, helpers=DL_HELPERS)


def _read(path):
    return read_json(path)


def _p1_line(p1) -> str:
    """One-line IDR Part-1 summary read from part1_idr.json, never assumed."""
    if not isinstance(p1, dict) or not p1:
        return "P1=PENDING FULL RUN (no idr/part1_idr.json)"
    primary = p1.get("primary") or {}
    rung = primary.get("rung_used")
    return (f"P1={primary.get('status', '?')} · window={p1.get('primary_window')} · "
            f"rung={rung if rung is not None else 'none'}")


def _p2_line(p2) -> str:
    """One-line IDR Part-2 summary read from part2_idr.json, never assumed."""
    if not isinstance(p2, dict) or not p2:
        return "P2=PENDING FULL RUN (no idr/part2_idr.json)"
    return f"P2={p2.get('status', '?')} · verdict={p2.get('verdict', 'not reached')}"


def stage_output_status(name: str) -> str:
    """Per-stage status read from the stage's own artefacts (never assumed)."""
    try:
        if name == "00_thresholds":
            obj = _read(THRESH)
            if obj is None:
                return "PENDING FULL RUN (no thresholds.json)"
            return (f"OK · provenance={obj.get('provenance', '?')} · "
                    f"frozen={obj.get('frozen')} · prereg_frozen={obj.get('prereg_frozen')}")
        if name == "01_audit":
            obj = _read(OUTPUTS / "audit" / "decision.json")
            if obj is None:
                return "PENDING FULL RUN (no audit/decision.json)"
            pending = ", ".join(obj.get("pending_items") or []) or "none"
            return f"{obj.get('gate', '?')} · pending items: {pending}"
        if name == "02_combine_ceiling":
            obj = _read(OUTPUTS / "ceiling" / "summary.json")
            if obj is None:
                return "PENDING FULL RUN (no ceiling/summary.json)"
            return (f"OK · {obj.get('n_players_with_fits', 0)} players · "
                    f"{obj.get('n_attempts_used', 0)} attempts · "
                    f"selected={obj.get('selected_features')}")
        if name == "03_game_snaps":
            obj = _read(OUTPUTS / "game" / "snaps" / "snaps_summary.json")
            if obj is None:
                return "PENDING FULL RUN (no game/snaps/snaps_summary.json)"
            return (f"OK · {obj.get('n_snaps', 0):,} snaps · "
                    f"event rate={obj.get('event_rate')} · reportable={obj.get('reportable')}")
        if name == "04_models":
            p1 = _read(OUTPUTS / "models" / "idr" / "part1_idr.json")
            p2 = _read(OUTPUTS / "models" / "idr" / "part2_idr.json")
            if p1 is None and p2 is None:
                return "PENDING FULL RUN (no models/idr/*.json)"
            return f"{_p1_line(p1)} · {_p2_line(p2)}"
        if name == "05_validate":
            obj = _read(OUTPUTS / "validation" / "validation_result.json")
            if obj is None:
                return "PENDING FULL RUN (no validation/validation_result.json)"
            perm = obj.get("rest_permutation") or {}
            rows = (obj.get("context_controls") or {}).get("rows") or []
            skipped = [r.get("check") for r in rows
                       if r.get("status") == "SKIPPED_BY_DECISION"]
            return (f"{obj.get('verdict', '?')} · rest_permutation={perm.get('status')} · "
                    f"skipped_by_decision={skipped or 'none'}")
    except Exception as exc:                # status printing must never kill the run
        return f"STATUS UNREADABLE ({type(exc).__name__}: {exc})"
    return "UNKNOWN STAGE"


_unknown_force = [s for s in FORCE_STAGES if s not in DL_STAGE_ORDER]
if _unknown_force:
    print(f"WARNING: FORCE_STAGES has unknown stage(s): {_unknown_force}")
RUN_ONLY_NORM = [str(s).strip() for s in (RUN_ONLY or []) if str(s).strip()]
_unknown_only = [s for s in RUN_ONLY_NORM if s not in DL_STAGE_ORDER]
if _unknown_only:
    print(f"WARNING: RUN_ONLY has unknown stage(s), ignored: {_unknown_only}")
STAGES_TO_RUN = [s for s in DL_STAGE_ORDER if (not RUN_ONLY_NORM or s in RUN_ONLY_NORM)]
SKIPPED_STAGES = [s for s in DL_STAGE_ORDER if s not in STAGES_TO_RUN]
if RUN_ONLY_NORM and not STAGES_TO_RUN:
    raise ValueError(f"RUN_ONLY={RUN_ONLY_NORM} matches no stage in DL_STAGE_ORDER")

# Belt and braces: the prereg cell already refused a full run on a DRAFT prereg.
if MODE == "full" and not globals().get("PREREG_FROZEN", False):
    raise RuntimeError("REFUSING TO RUN: MODE='full' while notes/prereg.md is not frozen "
                       "(see section 3). No stage will run.")

if SKIPPED_STAGES:
    print(f"RUN_ONLY={RUN_ONLY_NORM} — running {len(STAGES_TO_RUN)}/{len(DL_STAGE_ORDER)} stages; "
          "the rest are SKIPPED and their checkpoints/outputs are reused:")
    for name in SKIPPED_STAGES:
        print(f"  [SKIPPED — checkpoint] {name}")
    print()

RUN_RESULTS = []
print(f"running DL chain in '{MODE}' mode — each stage reloads its code from disk ...\n")
for name in STAGES_TO_RUN:
    force = ["--force"] if (FORCE or name in FORCE_STAGES) else []

    # Never run a full-data stage on the truncated sample: it could stamp FULL
    # provenance from sample-sized input. Report the missing files instead.
    if MODE == "full" and name in FULL_DATA_STAGES and not FULL_TRACKING_READY:
        reason = "missing full tracking: " + ", ".join(FULL_TRACKING_MISSING)
        res = {"stage": name, "ok": False, "seconds": 0.0, "peak_mb": None,
               "status": "PENDING FULL RUN", "error": reason, "tail": [], "crash_file": None}
        RUN_RESULTS.append(res)
        print(f"[{'PENDING FULL RUN':<44}] {name:<18} {'—':>8}   {reason}")
        continue

    offset = _runlog_rows()
    try:
        res = _run_dl_stage(name, ["--mode", MODE, *force])
    except BaseException as exc:             # load-time failure: keep the loop alive
        import traceback
        res = {"stage": name, "ok": False, "seconds": 0.0, "peak_mb": None,
               "error": f"{type(exc).__name__}: {exc}", "tail": [],
               "traceback": traceback.format_exc(), "crash_file": None}
    res["peak_mb"] = _peak_mb_after(offset, name)

    if res.get("ok"):
        res["status"] = stage_output_status(name)
    else:
        error = str(res.get("error") or "")
        missing_input = ("FileNotFoundError" in error or "No such file" in error
                         or "not found" in error.lower())
        if MODE == "full" and name in FULL_DATA_STAGES and missing_input:
            res["status"] = "PENDING FULL RUN"
        else:
            res["status"] = "FAILED"
    RUN_RESULTS.append(res)

    ram = f"{res['peak_mb']:.1f} MB" if res.get("peak_mb") is not None else "—"
    print(f"[{res['status']:<44}] {name:<18} {res['seconds']:>7.2f}s  peak RAM {ram}")
    if not res.get("ok"):
        for line in (res.get("tail") or [])[-5:]:
            print(f"           {line}")
        if res.get("crash_file"):
            print(f"           crash report: {res['crash_file']}")

print()
print("per-stage result:")
for res in RUN_RESULTS:
    ram = f"{res['peak_mb']:.1f} MB" if res.get("peak_mb") is not None else "—"
    print(f"  {res['stage']:<18} {res['status']:<20} {res['seconds']:>7.2f}s  {ram}")
print("\n(Per-stage statuses come from the stages' own artefacts; a skipped or PENDING stage "
      "is not a failure.)")

## 5. Headline — the block to send back

Every value below is read from a file under `outputs/dl/`; none is typed here. A missing file or
column prints `NOT RUN — <field> absent`, and a non-`FULL` provenance is never reported as a
finding.

It prints, in this order:

1. **P1** — the load × rest interaction on `log(deficit_ratio)` with its interval, the **rung used**
   from the declared convergence ladder, and the declared sensitivity windows (1 and 5 snaps).
2. **P2** — the deficit-vs-absolute held-out log-score comparison with its interval, verdict and
   smallest effect of interest.
3. **Phase 5** — the within-player rest permutation (`p`, null summary, singleton blocks) and every
   context control with its status; `SKIPPED_BY_DECISION` is printed with its reason.
4. **Coverage / measurement counters** — rows tracked, fits OK, `rest_sequence_complete`, singleton
   blocks, and how many rows carry load / baseline / deficit.

**In `MODE="sample"` the estimates are `WITHHELD`.** Statuses, reasons and coverage counters are
printed, because whether something *can* be estimated is a property of coverage, not a result. No
sample number is printed.

In [ ]:
import math

print("HEADLINE — Intra-drive regeneration: per-snap kinematic deficits → pass rush")
print("=" * 90)

if MODE != "full":
    print("WITHHELD — MODE='sample'.")
    print("Reason: every artefact produced in sample mode is stamped")
    print("        'PLACEHOLDER_SAMPLE: UNVALIDATED SAMPLE OUTPUT'.")
    print("        No sample number may be quoted as a result, so none is printed here.")
    print("        Set MODE='full' with a frozen prereg to obtain the pre-declared verdict.")
    print("")
    print("STATUS AND ESTIMABILITY — structure only, absolutely no estimates (not a finding):")

    p1_s = read_json(OUTPUTS / "models" / "idr" / "part1_idr.json")
    if p1_s is None:
        print("  P1: NOT RUN — outputs/dl/models/idr/part1_idr.json absent")
    else:
        prim = p1_s.get("primary") or {}
        frame = prim.get("frame") or {}
        print(f"  P1 ({p1_s.get('provenance')}): {prim.get('status')} — "
              f"{prim.get('reason') or 'no reason recorded'}")
        print(f"     window {p1_s.get('primary_window')} · rows in {frame.get('n_rows_in')} · "
              f"complete {frame.get('n_complete')} · players {frame.get('n_players')} · "
              f"rung used {prim.get('rung_used')}")
        for row in (p1_s.get("panel") or []):
            if row.get("window") == p1_s.get("primary_window"):
                continue
            frow = row.get("frame") or {}
            print(f"     sensitivity window {row.get('window')}: {row.get('status')} — "
                  f"{row.get('reason') or 'no reason recorded'} "
                  f"(complete {frow.get('n_complete')})")

    p2_s = read_json(OUTPUTS / "models" / "idr" / "part2_idr.json")
    if p2_s is None:
        print("  P2: NOT RUN — outputs/dl/models/idr/part2_idr.json absent")
    else:
        cov = p2_s.get("coverage") or {}
        print(f"  P2 ({p2_s.get('provenance')}): {p2_s.get('status')} — "
              f"{p2_s.get('reason') or 'no reason recorded'}")
        print(f"     complete-case rows {cov.get('n_complete_case_rows')} · with deficit "
              f"{cov.get('n_with_deficit')} · with absolute {cov.get('n_with_absolute')} · "
              f"with both {cov.get('n_with_both')}")

    val_s = read_json(OUTPUTS / "validation" / "validation_result.json")
    if val_s is None:
        print("  Phase 5: NOT RUN — outputs/dl/validation/validation_result.json absent")
    else:
        perm = val_s.get("rest_permutation") or {}
        des = perm.get("design") or {}
        print(f"  Phase 5 ({val_s.get('provenance')}): {val_s.get('verdict')} — "
              f"{val_s.get('verdict_statement')}")
        print(f"     within-player rest permutation: {perm.get('status')} — "
              f"{perm.get('reason') or 'no reason recorded'}")
        print(f"     blocks {des.get('n_blocks')} · singleton {des.get('n_blocks_singleton')} · "
              f"snaps in singleton blocks {des.get('n_snaps_in_singleton_blocks')}")
        for row in ((val_s.get("context_controls") or {}).get("rows") or []):
            print(f"     [{row.get('status')}] {row.get('check')}: "
                  f"{row.get('reason') or 'no reason recorded'}")
    print("")
    print("  every estimate stays withheld: sample output is UNVALIDATED.")
else:
    def _f(value, digits=4):
        if value is None:
            return "NA"
        try:
            f = float(value)
        except (TypeError, ValueError):
            return str(value)
        return f"{f:.{digits}f}" if math.isfinite(f) else "NA"

    def _why(obj):
        if not isinstance(obj, dict):
            return ""
        reason = obj.get("reason")
        if isinstance(reason, dict):
            return str(reason.get("message") or reason.get("code") or "")
        return "" if reason is None else str(reason)

    def _artifact(path, label):
        """A JSON artefact that may be reported; None (with a NOT RUN line) otherwise."""
        obj = read_json(path)
        if obj is None:
            print(f"  NOT RUN — {label} absent")
            return None
        prov = str(obj.get("provenance", ""))
        if prov != "FULL":
            print(f"  NOT RUN — {label} provenance is {prov!r}, not 'FULL'")
            return None
        return obj

    # ---- 1. P1 -----------------------------------------------------------------
    print("\n1. P1 — rolling load × inter-snap rest → acceleration deficit")
    print("   source: outputs/dl/models/idr/part1_idr.json")
    p1 = _artifact(OUTPUTS / "models" / "idr" / "part1_idr.json",
                   "outputs/dl/models/idr/part1_idr.json")
    if p1 is None:
        print("  P1 NOT RUN — no reportable Part-1 artefact.")
    else:
        prim = p1.get("primary") or {}
        print(f"  window: prior {p1.get('primary_window')} snaps "
              f"(declared primary; sensitivity {p1.get('sensitivity_windows')}); "
              f"outcome {p1.get('outcome')}")
        if prim.get("status") != "OK":
            print(f"  NOT ESTIMABLE — {_why(prim) or 'no reason recorded'}")
            frame = prim.get("frame") or {}
            print(f"  frame: rows in {frame.get('n_rows_in')} · complete "
                  f"{frame.get('n_complete')} · players {frame.get('n_players')} · "
                  f"games {frame.get('n_games')}")
        else:
            inter = prim.get("interaction") or {}
            contrast = prim.get("contrast") or {}
            print(f"  interaction (load × rest, both standardised): "
                  f"{_f(inter.get('coef_std'), 5)}")
            print(f"    95% CI [{_f(inter.get('ci_low_std'), 5)}, "
                  f"{_f(inter.get('ci_high_std'), 5)}] · p = {_f(inter.get('p'), 4)} · "
                  f"player-clustered SE {_f(inter.get('se_std'), 5)}")
            print(f"    rung used: {prim.get('rung_used')} "
                  f"(declared ladder: {prim.get('rungs_intended')})")
            print(f"    contrast: slope of log-deficit on load, +1 SD rest minus −1 SD rest "
                  f"= {_f(contrast.get('difference'), 5)}")
            print(f"    {'NEGATIVE' if (contrast.get('difference') or 0) < 0 else 'POSITIVE'} "
                  f"means load hurts the deficit "
                  f"{'more' if (contrast.get('difference') or 0) < 0 else 'less'} when rest is "
                  f"{'short' if (contrast.get('difference') or 0) < 0 else 'long'}")
            print(f"    n snaps {prim.get('n_snaps')} · players {prim.get('n_players')} · "
                  f"games {prim.get('n_games')} · cluster {inter.get('cluster')} "
                  f"(dof {inter.get('dof')})")
            print("    P1 declared status: an estimate with a 95% interval — no "
                  "threshold, no confirmatory label (D85; prereg §12)")
            _p1_rule = globals().get("PREREG_P1_RULE_TEXT", "")
            if _p1_rule and "not restated verbatim" not in _p1_rule:
                print(f"    prereg: {_p1_rule}")
            for rung in (prim.get("rung") or prim.get("ladder") or []):
                print(f"      ladder: {rung.get('rung')} → {rung.get('status')}"
                      + (f" ({rung.get('reason')})" if rung.get("reason") else ""))
        print("  sensitivity panel (declared, never selected on data):")
        for row in (p1.get("panel") or []):
            if row.get("window") == p1.get("primary_window"):
                continue
            frame = row.get("frame") or {}
            if row.get("status") == "OK":
                inter = row.get("interaction") or {}
                print(f"    window {row.get('window')}: OK · coef "
                      f"{_f(inter.get('coef_std'), 5)} "
                      f"[{_f(inter.get('ci_low_std'), 5)}, {_f(inter.get('ci_high_std'), 5)}] · "
                      f"rung {row.get('rung_used')} · complete {frame.get('n_complete')}")
            else:
                print(f"    window {row.get('window')}: {row.get('status')} — "
                      f"{_why(row) or 'no reason recorded'} "
                      f"(complete {frame.get('n_complete')})")

    # ---- 2. P2 -----------------------------------------------------------------
    print("\n2. P2 — snap-level deficit vs absolute cumulative load (out of sample)")
    print("   source: outputs/dl/models/idr/part2_idr.json")
    p2 = _artifact(OUTPUTS / "models" / "idr" / "part2_idr.json",
                   "outputs/dl/models/idr/part2_idr.json")
    if p2 is None:
        print("  P2 NOT RUN — no reportable Part-2 artefact.")
    elif p2.get("status") != "OK":
        print(f"  NOT ESTIMABLE — {_why(p2) or 'no reason recorded'}")
        cov = p2.get("coverage") or {}
        folds = p2.get("fold_info") or {}
        print(f"  coverage: complete-case rows {cov.get('n_complete_case_rows')} · "
              f"with deficit {cov.get('n_with_deficit')} · "
              f"with absolute {cov.get('n_with_absolute')} · "
              f"with both {cov.get('n_with_both')}")
        print(f"  folds: {folds.get('folds')} of {folds.get('requested')} requested"
              + (f" ({folds.get('reason')})" if folds.get("reason") else ""))
        print(f"  metric: {p2.get('metric')} · sign: {p2.get('delta_sign')}")
    else:
        scores = p2.get("mean_logscore") or {}
        lo, hi = (p2.get("difference_interval") or [None, None])[:2]
        print(f"  metric: {p2.get('metric')} (nats/snap; higher is better)")
        print(f"  deficit arm {_f(scores.get('deficit'), 5)} · "
              f"absolute arm {_f(scores.get('absolute'), 5)}")
        print(f"  difference (deficit − absolute): {_f(p2.get('difference_nats_per_snap'), 5)} "
              f"nats/snap · interval [{_f(lo, 5)}, {_f(hi, 5)}]")
        print(f"  verdict: {p2.get('verdict')} · better arm: {p2.get('better_arm')} · "
              f"SOI {_f(p2.get('soi'), 5)} · equivalence margin "
              f"{_f(p2.get('equivalence_margin'), 5)}")
        print(f"  excludes zero: {p2.get('excludes_zero')} · exceeds SOI: "
              f"{p2.get('exceeds_soi')} · scored snaps {p2.get('n_scored_snaps')} · "
              f"players {p2.get('n_scored_players')}")
        folds = p2.get("fold_info") or {}
        print(f"  folds: {folds.get('folds')} of {folds.get('requested')} requested"
              + (f" ({folds.get('reason')})" if folds.get("reason") else ""))

    # ---- 3. Phase 5 -------------------------------------------------------------
    print("\n3. Phase 5 — within-player rest permutation and context controls")
    print("   source: outputs/dl/validation/validation_result.json")
    validation = read_json(OUTPUTS / "validation" / "validation_result.json")
    if validation is None:
        print("  NOT RUN — outputs/dl/validation/validation_result.json absent")
    elif str(validation.get("provenance", "")) != "FULL":
        print(f"  NOT RUN — validation provenance is "
              f"{validation.get('provenance')!r}, not 'FULL'")
    else:
        print(f"  verdict: {validation.get('verdict')}")
        print(f"  {validation.get('verdict_statement')}")

        perm = validation.get("rest_permutation") or {}
        des = perm.get("design") or {}
        obs = perm.get("observed") or {}
        inf = perm.get("inference") or {}
        if not perm:
            print("  NOT RUN — rest permutation absent from validation_result.json")
        elif perm.get("status") != "OK":
            print(f"  [{perm.get('status')}] within-player rest permutation: "
                  f"{perm.get('reason') or 'no reason recorded'}")
            print(f"       design: blocks {des.get('n_blocks')} · singleton blocks "
                  f"{des.get('n_blocks_singleton')} · snaps in singleton blocks "
                  f"{des.get('n_snaps_in_singleton_blocks')} · valid refits "
                  f"{des.get('reps_valid')}/{des.get('reps_requested')}")
        else:
            print(f"  [OK] within-player rest permutation: observed "
                  f"{_f(obs.get('coef_std'), 5)} · null mean {_f(inf.get('null_mean'), 5)} · "
                  f"SD {_f(inf.get('null_sd'), 5)}")
            print(f"       p(two-sided) {_f(inf.get('p_two_sided'), 4)} · "
                  f"p(one-sided) {_f(inf.get('p_one_sided'), 4)} · "
                  f"{inf.get('n_permutations')} valid refits "
                  f"(requested {des.get('reps_requested')}, seed {des.get('seed')}) · "
                  f"reject null at alpha {_f(inf.get('alpha'), 3)}: {inf.get('reject_null')}")
            print(f"       blocks {des.get('n_blocks')} · permuted {des.get('n_blocks_permuted')} · "
                  f"singleton blocks {des.get('n_blocks_singleton')} · snaps in singleton "
                  f"blocks {des.get('n_snaps_in_singleton_blocks')} (counted, never moved) · "
                  f"multiset preserved {des.get('block_multiset_preserved')}")

        ctx = validation.get("context_controls") or {}
        if not (ctx.get("rows") or []):
            print("  NOT RUN — context controls absent from validation_result.json")
        else:
            print("  context controls (same ladder, restricted sample):")
            for row in ctx["rows"]:
                est = row.get("estimate") or {}
                detail = ""
                if row.get("status") == "OK" and est.get("coef_std") is not None:
                    detail = (f"interaction {_f(est.get('coef_std'), 5)} · "
                              f"n = {row.get('n_rows')} · excluded {row.get('n_excluded')}")
                else:
                    detail = (f"n = {row.get('n_rows')} · excluded {row.get('n_excluded')}"
                              + (f" · {row.get('reason')}" if row.get("reason") else ""))
                print(f"    [{row.get('status')}] {row.get('check')}: {detail}")

        ceil = validation.get("ceiling_permutation") or {}
        print(f"  [SUPERSEDED] ceiling permutation: {ceil.get('status')} — "
              f"{ceil.get('reason') or ''}")
        missing = validation.get("not_implemented") or []
        print(f"  NOT IMPLEMENTED in this build ({len(missing)}): "
              f"{', '.join(missing) if missing else 'none'}")

    # ---- 4. coverage and measurement counters ----------------------------------
    print("\n4. COVERAGE AND MEASUREMENT COUNTERS")
    print("   source: outputs/dl/game/snaps/snaps_summary.json (phase3 block)")
    snaps_summary = _artifact(OUTPUTS / "game" / "snaps" / "snaps_summary.json",
                              "outputs/dl/game/snaps/snaps_summary.json")
    if snaps_summary is None:
        print("  NOT RUN — no reportable snap summary.")
    else:
        ph3 = snaps_summary.get("phase3") or {}
        fit = ph3.get("fit") or {}
        rest = ph3.get("rest") or {}
        load = ph3.get("load") or {}
        base = ph3.get("baseline") or {}
        n_rows = ph3.get("n_rows", snaps_summary.get("n_snaps"))
        print(f"  snaps in the table: {n_rows:,}" if isinstance(n_rows, int)
              else f"  snaps in the table: {n_rows}")
        if not fit:
            print("  NOT RUN — phase3.fit counters absent")
        else:
            print(f"  rows tracked      : {fit.get('n_rows_rusher_tracked')} "
                  f"({_f((fit.get('share_rows_rusher_tracked') or 0) * 100, 2)}% of rows)")
            print(f"  fits OK           : {fit.get('n_fit_ok')} of "
                  f"{fit.get('n_rows_rusher_tracked')} tracked "
                  f"({_f((fit.get('share_fit_ok_of_tracked') or 0) * 100, 2)}%) · "
                  f"at bound {fit.get('n_fit_at_bound')} · "
                  f"failed/low R² {fit.get('n_fit_failed_or_low_r2')} · "
                  f"mean R² {_f(fit.get('mean_fit_r2'), 3)}")
        if not rest:
            print("  NOT RUN — phase3.rest counters absent")
        else:
            print(f"  rest_sequence_complete: {rest.get('n_player_games_complete')} of "
                  f"{rest.get('n_player_games')} player-games "
                  f"({_f((rest.get('share_player_games_complete') or 0) * 100, 2)}% complete) · "
                  f"rows in incomplete sequence {rest.get('n_rows_in_incomplete_sequence')}")
            print(f"  inter_snap_rest_s present: {rest.get('n_rest_present')} rows "
                  f"({_f((rest.get('share_rest_present') or 0) * 100, 2)}%) · "
                  f"first-snap NA {rest.get('n_first_snap_na')} · "
                  f"prev play without end timestamp "
                  f"{rest.get('n_prev_play_without_end_timestamp')}")
        if load:
            print(f"  rolling load      : {load.get('n_rolling_present')} rows with a complete "
                  f"{load.get('window_snaps')}-snap window · incomplete "
                  f"{load.get('n_window_incomplete')} · contains excluded snap "
                  f"{load.get('n_window_contains_excluded_snap')}")
        if base:
            print(f"  baseline / deficit: {base.get('n_rows_with_baseline')} rows with an a0 "
                  f"baseline ({base.get('n_players_with_baseline')} players) · "
                  f"{base.get('n_deficit_computed')} rows with a deficit_ratio")
        print(f"  singleton blocks  : from Phase 5 above — a block with one snap cannot be "
              f"permuted and is counted, never dropped")

    # ---- 5. prereg reminder -----------------------------------------------------
    print("\n5. PREREG STATUS (the gate that governs the run)")
    print(f"  notes/prereg.md declared status: {_status_line or 'NO STATUS LINE FOUND'}")
    print(f"  frozen: {PREREG_FROZEN}")
    print(f"  {PREREG_P1_RULE_TEXT}")

print("\n" + "=" * 90)
print("End of headline. Missing items say NOT RUN; they are never filled with a guess or a zero.")

## 6. What to send back

If the run stops or something looks wrong, these are the artefacts that matter — send the files
themselves, not screenshots of the terminal:

1. **The headline cell output** (§5). It is the plain-English verdict.
2. `outputs/dl/audit/decision.json` and `outputs/dl/audit/SUMMARY.md` — the go/no-go gate and every
   audit item with its evidence (`i_icc.csv` for the `a0` test-retest, `c_snap_coverage.csv` for
   rest continuity, `d_null_semantics.csv`).
3. `outputs/dl/ceiling/summary.json` and `player_ceiling.csv` — the shrunken `a0` baseline, its SE
   and the ICC/VIF screen.
4. `outputs/dl/game/snaps/snaps_summary.json` — snap counts, exclusions, and the `phase3` block with
   `rest`, `fit`, `load` and `baseline` counters (tracked rows, fits OK, `rest_sequence_complete`).
5. `outputs/dl/models/idr/part1_idr.json` — the P1 interaction, its CI, the rung ladder and the
   sensitivity panel; `outputs/dl/models/idr/part2_idr.json` — the P2 comparison or its reason.
6. `outputs/dl/validation/validation_result.json`, `rest_permutation_null.csv`,
   `context_controls.csv` — the permutation null, its `p`, and every context control (including
   `SKIPPED_BY_DECISION`).
7. `outputs/run_log.csv` (the DL rows) — wall time and peak RAM per stage.
8. If a stage is `FAILED`: the `crash_*.txt` path printed in §4.
9. If the run was refused at §3: the printed prereg status block, plus `notes/prereg.md`.
10. `notes/blockers.md` — every open data/design gap, including B-DL-IDR-01 (obvious passing
    situations undefined and skipped by decision).

## 7. Diagnostic figures (≤2, full mode only)

Rendered only from `outputs/dl/` and only in `MODE="full"`. In sample mode they are skipped with a
reason: a plot of sample output is not a finding. If the needed columns are absent, or provenance is
not `FULL`, the figure is skipped without an exception. These are diagnostics, not writeup figures.

* **Figure 1** — the P1 shape: mean `deficit_ratio` against rolling kinematic load, split by rest
  tertile. A downward slope that is steeper for short rest is the fatigue interaction made visible.
* **Figure 2** — the premise: shrunken Combine `a0` baseline against each player's mean in-game
  deficit (player level). It shows whether the baseline and the exposure are doing different work.

In [ ]:
if not SHOW_FIGURES:
    print("figures: SHOW_FIGURES=False — skipped")
elif MODE != "full":
    print("figures: SKIPPED — MODE='sample'. A plot of sample output would not be a finding.")
else:
    try:
        import matplotlib
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f"figures: SKIPPED — matplotlib unavailable ({type(exc).__name__}: {exc})")
    else:
        made = 0
        SNAP_COLS = {"provenance", "nfl_id", "deficit_ratio",
                     "rolling_kinematic_load_w_per_kg", "inter_snap_rest_s"}

        # Figure 1 — the P1 shape: deficit vs load, split by rest tertile.
        snaps = read_csv(OUTPUTS / "game" / "snaps" / "snaps.csv",
                         usecols=lambda c: c in SNAP_COLS)
        if not len(snaps):
            print("figure 1: SKIPPED — outputs/dl/game/snaps/snaps.csv absent")
        elif prov_of(snaps) != "FULL":
            print(f"figure 1: SKIPPED — snaps provenance is {prov_of(snaps)!r}, not 'FULL'")
        else:
            need = {"deficit_ratio", "rolling_kinematic_load_w_per_kg", "inter_snap_rest_s"}
            if not need.issubset(snaps.columns):
                print("figure 1: SKIPPED — deficit_ratio / rolling load / rest column absent")
            else:
                work = snaps[list(need)].apply(to_num).replace([float("inf"), float("-inf")], float("nan"))
                work = work.dropna()
                if len(work) < 50:
                    print(f"figure 1: SKIPPED — only {len(work)} complete rows")
                else:
                    try:
                        fig, ax = plt.subplots(figsize=(8, 4.4))
                        work["rest_band"] = pd.qcut(
                            work["inter_snap_rest_s"], 3, labels=["short", "mid", "long"],
                            duplicates="drop")
                        for label, group in work.groupby("rest_band", observed=True):
                            if len(group) < 20:
                                print(f"figure 1: band {label} SKIPPED — {len(group)} rows")
                                continue
                            bins = pd.qcut(group["rolling_kinematic_load_w_per_kg"], 8,
                                           duplicates="drop")
                            curve = (group.groupby(bins, observed=True)
                                     .agg(load=("rolling_kinematic_load_w_per_kg", "mean"),
                                          deficit=("deficit_ratio", "mean"),
                                          n=("deficit_ratio", "size")))
                            ax.plot(curve["load"], curve["deficit"], marker="o",
                                    label=f"{label} rest (n={len(group)})")
                        ax.set_xlabel(f"rolling kinematic load before snap (W/kg, prior "
                                      f"{LOAD_WINDOW} snaps)")
                        ax.set_ylabel("mean deficit_ratio (snap a / a0)")
                        ax.set_title("P1 shape — deficit vs load by rest tertile (FULL diagnostic)")
                        ax.legend()
                        path = FIGDIR / "dl_idr_fig1_load_rest_deficit.png"
                        fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
                        plt.show()
                        print(f"figure 1: saved -> {path}")
                        made += 1
                    except Exception as exc:
                        print(f"figure 1: SKIPPED — {type(exc).__name__}: {exc}")

        # Figure 2 — the premise: Combine baseline a0 vs mean in-game deficit.
        ceiling = read_csv(OUTPUTS / "ceiling" / "player_ceiling.csv")
        if not len(ceiling) or not len(snaps):
            print("figure 2: SKIPPED — player_ceiling.csv or snaps.csv absent")
        elif prov_of(ceiling) != "FULL" or prov_of(snaps) != "FULL":
            print("figure 2: SKIPPED — ceiling/snaps provenance is not FULL")
        elif "deficit_ratio" not in snaps.columns or "nfl_id" not in snaps.columns:
            print("figure 2: SKIPPED — deficit_ratio absent from snaps.csv")
        else:
            try:
                a0 = ceiling[ceiling["feature"] == "a0"][["nfl_id", "estimate"]].copy()
                per_player = (snaps.assign(deficit=to_num(snaps["deficit_ratio"]))
                              .groupby("nfl_id", as_index=False)["deficit"].mean())
                merged = a0.merge(per_player, on="nfl_id", how="inner").dropna()
                if len(merged) < 5:
                    print(f"figure 2: SKIPPED — only {len(merged)} players with both a0 and a deficit")
                else:
                    fig, ax = plt.subplots(figsize=(7.5, 4.4))
                    ax.scatter(to_num(merged["estimate"]), to_num(merged["deficit"]),
                               s=18, alpha=0.7)
                    ax.set_xlabel("shrunken Combine a0 (yd/s²) — the baseline")
                    ax.set_ylabel("mean in-game deficit_ratio — the exposure")
                    ax.set_title("Baseline vs exposure, player level (FULL diagnostic)")
                    path = FIGDIR / "dl_idr_fig2_baseline_vs_deficit.png"
                    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
                    plt.show()
                    print(f"figure 2: saved -> {path}")
                    made += 1
            except Exception as exc:
                print(f"figure 2: SKIPPED — {type(exc).__name__}: {exc}")

        print(f"figures made: {made} (diagnostics only, not writeup figures)")